# 05 — Evaluation & Comparison
Load all trained models and compare them across metrics.

In [ ]:
# --- Google Colab Setup ---
import os

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -e ".[notebook]" --no-build-isolation
    # Apply NumPy 2.x / API compatibility patches
    import site; SITE = site.getsitepackages()[0]
    !patch -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch
    !patch -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch
    !patch -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch
    !patch -p1 {SITE}/gym/wrappers/time_limit.py       < patches/gym_time_limit_compat.patch
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
# Load models
models = {
    'Pixel DQN': (DQN.load('../models/pixel_dqn/final_model'), make_pixel_env),
    'Pixel PPO': (PPO.load('../models/pixel_ppo/final_model'), make_pixel_env),
    'Symbolic DQN': (DQN.load('../models/symbolic_dqn/final_model'), make_symbolic_env),
    'Symbolic PPO': (PPO.load('../models/symbolic_ppo/final_model'), make_symbolic_env),
}

In [ ]:
# Evaluate all models
all_results = {}
for name, (model, env_fn) in models.items():
    print(f'Evaluating {name}...')
    env = env_fn()
    results = evaluate_agent(model, env, n_episodes=EVAL_DEFAULTS.n_eval_episodes)
    all_results[name] = results
    print(f"  Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
    print(f"  Flag rate: {results['flag_rate']:.2%}")
    env.close()

In [ ]:
# Summary table
df = pd.DataFrame({
    name: {
        'Mean Reward': f"{r['mean_reward']:.1f} ± {r['std_reward']:.1f}",
        'Mean Length': f"{r['mean_length']:.0f}",
        'Flag Rate': f"{r['flag_rate']:.1%}",
    }
    for name, r in all_results.items()
}).T
df

In [ ]:
# Save results
df.to_csv('../results/evaluation_summary.csv')
print('Results saved to ../results/evaluation_summary.csv')